In [0]:
import urllib.request

# ── 1. Create the volume ────────────────────────────────────────────────────
spark.sql("CREATE VOLUME IF NOT EXISTS sales.mapping.mappings")
print("Volume sales.mapping.mappings is ready.")

# ── 2. Download Excel file from GitHub into the volume ──────────────────────
GITHUB_URL  = (
    "https://raw.githubusercontent.com/marvinjayson/DATABRICKS-END-TO-END"
    "/main/mapping/master_mapping_bronze_silver_gold_pii.xlsx"
)
DEST_PATH   = "/Volumes/sales/mapping/mappings/master_mapping_bronze_silver_gold_pii.xlsx"

urllib.request.urlretrieve(GITHUB_URL, DEST_PATH)
print(f"File saved to: {DEST_PATH}")

# ── 3. Verify ────────────────────────────────────────────────────────────────
import os
size_kb = os.path.getsize(DEST_PATH) / 1024
print(f"File size: {size_kb:.1f} KB")

In [0]:
%pip install openpyxl pyyaml -q

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

FILE_PATH = "/Volumes/sales/bronze/mappings/master_mapping_bronze_silver_gold_pii.xlsx"

# List all sheets
xl = pd.ExcelFile(FILE_PATH)
print("Sheets:", xl.sheet_names)

# Preview each sheet
for sheet in xl.sheet_names:
    df = xl.parse(sheet)
    print(f"\n--- Sheet: '{sheet}' ({df.shape[0]} rows x {df.shape[1]} cols) ---")
    print(df.head(3).to_string())

In [0]:
import pandas as pd
import yaml
from collections import defaultdict

FILE_PATH   = "/Volumes/sales/mapping/mappings/master_mapping_bronze_silver_gold_pii.xlsx"
YAML_OUTPUT = "/Volumes/sales/mapping/mappings/pipeline_config.yaml"

def clean_df(sheet_name):
    """Read a sheet (header is row index 3 — rows 0-2 are title/subtitle/blank) and drop all-NaN rows."""
    df = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=3)
    return df.dropna(how="all").reset_index(drop=True)

def strip_nan(d: dict) -> dict:
    """Remove keys whose value is NaN / empty string."""
    return {
        k: str(v).strip()
        for k, v in d.items()
        if pd.notna(v) and str(v).strip() not in ("", "nan", "None")
    }

# ── Bronze → Silver ──────────────────────────────────────────────────────────
b2s = clean_df("Bronze_to_Silver")
b2s_map = defaultdict(lambda: {"source_table": None, "columns": []})
for _, row in b2s.iterrows():
    target = row.get("Target Table")
    if pd.isna(target):
        continue
    b2s_map[target]["source_table"] = row.get("Source Table")
    b2s_map[target]["columns"].append(strip_nan({
        "source_column":      row.get("Source Column"),
        "target_column":      row.get("Target Column"),
        "target_data_type":   row.get("Target Data Type"),
        "transformation":     row.get("Transformation Logic"),
        "dq_rule":            row.get("DQ / Filter Rule"),
        "business_definition":row.get("Business Definition"),
        "status":             row.get("Status"),
        "pii_tag":            row.get("PII Tag"),
    }))

# ── Silver → Gold ────────────────────────────────────────────────────────────
s2g = clean_df("Silver_to_Gold")
s2g_map = defaultdict(lambda: {"source_tables": None, "columns": []})
for _, row in s2g.iterrows():
    target = row.get("Target Gold Table")
    if pd.isna(target):
        continue
    s2g_map[target]["source_tables"] = row.get("Source Table(s)")
    s2g_map[target]["columns"].append(strip_nan({
        "source_columns":  row.get("Source Column(s)"),
        "target_column":   row.get("Target Column"),
        "target_data_type":row.get("Target Data Type"),
        "transformation":  row.get("Aggregation / Transformation Logic"),
        "grain":           row.get("Grain"),
        "business_use":    row.get("Business Use"),
        "status":          row.get("Status"),
        "pii_tag":         row.get("PII Tag"),
    }))

# ── DQ Rules ─────────────────────────────────────────────────────────────────
dq = clean_df("DQ_Rules")
dq_map = defaultdict(list)
for _, row in dq.iterrows():
    if pd.isna(row.get("Rule ID")):
        continue
    table = str(row.get("Table", "unknown")).strip()
    dq_map[table].append(strip_nan({
        "rule_id":    row.get("Rule ID"),
        "layer":      row.get("Layer"),
        "columns":    row.get("Column(s)"),
        "rule_type":  row.get("Rule Type"),
        "expectation":row.get("Expectation / Rule"),
        "action":     row.get("Action"),
        "severity":   row.get("Severity"),
        "status":     row.get("Status"),
        "pii_tag":    row.get("PII Tag"),
    }))

# ── Assemble & write YAML ────────────────────────────────────────────────────
config = {
    "pipeline": {
        "name": "olist_lakehouse",
        "description": "Olist e-commerce medallion pipeline — Bronze to Silver to Gold",
        "bronze_to_silver": {
            t: {"source_table": str(m["source_table"]), "columns": m["columns"]}
            for t, m in b2s_map.items()
        },
        "silver_to_gold": {
            t: {"source_tables": str(m["source_tables"]), "columns": m["columns"]}
            for t, m in s2g_map.items()
        },
        "dq_rules": dict(dq_map),
    }
}

with open(YAML_OUTPUT, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

import os
size_kb = os.path.getsize(YAML_OUTPUT) / 1024
with open(YAML_OUTPUT) as f:
    lines = f.readlines()

print(f"YAML written to : {YAML_OUTPUT}")
print(f"File size       : {size_kb:.1f} KB  ({len(lines)} lines)")
print(f"Bronze->Silver  : {len(b2s_map)} target tables")
print(f"Silver->Gold    : {len(s2g_map)} target tables")
print(f"DQ rule groups  : {len(dq_map)} tables\n")
print("--- Preview (first 60 lines) ---")
print("".join(lines[:60]))

In [0]:
import yaml

BASE = "/Volumes/sales/mapping/mappings"

with open(f"{BASE}/pipeline_config.yaml") as f:
    config = yaml.safe_load(f)["pipeline"]

# Extract table lists per medallion layer from the master config
bronze = sorted({v["source_table"] for v in config["bronze_to_silver"].values() if v.get("source_table")})
silver = list(config["bronze_to_silver"].keys())
gold   = list(config["silver_to_gold"].keys())

out = {
    "tables": {"BRONZE": bronze, "SILVER": silver, "GOLD": gold},
    "output_table": "sales.audit.row_count_audit",
}

with open(f"{BASE}/audit_tables.yaml", "w") as f:
    yaml.dump(out, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"audit_tables.yaml written  ✓")
print(f"  BRONZE : {len(bronze)} tables")
print(f"  SILVER : {len(silver)} tables")
print(f"  GOLD   : {len(gold)} tables")

In [0]:
import yaml

BASE = "/Volumes/sales/mapping/mappings"

with open(f"{BASE}/pipeline_config.yaml") as f:
    config = yaml.safe_load(f)["pipeline"]

# Preserve manually-written fail_sql from the existing file
try:
    with open(f"{BASE}/dq_rules.yaml") as f:
        existing_rules = {r["rule_id"]: r for r in yaml.safe_load(f).get("rules", [])}
except FileNotFoundError:
    existing_rules = {}

# Flatten dq_rules from pipeline_config (grouped by table → flat list)
rules = []
for table, table_rules in config.get("dq_rules", {}).items():
    for rule in table_rules:
        rule_id  = rule.get("rule_id", "")
        existing = existing_rules.get(rule_id, {})
        rules.append({
            "rule_id":     rule_id,
            "layer":       rule.get("layer", ""),
            "table":       table,
            "rule_type":   rule.get("rule_type", ""),
            "severity":    rule.get("severity", ""),
            "expectation": rule.get("expectation", ""),
            # Preserve existing fail_sql; leave placeholder for new rules
            "fail_sql":    existing.get("fail_sql", f"-- TODO: add fail_sql for {rule_id}"),
        })

out = {"output_table": "sales.dq.dq_results", "rules": rules}

with open(f"{BASE}/dq_rules.yaml", "w") as f:
    yaml.dump(out, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

todos = [r["rule_id"] for r in rules if "TODO" in r.get("fail_sql", "")]
print(f"dq_rules.yaml written  ✓  ({len(rules)} rules)")
if todos:
    print(f"  ⚠  {len(todos)} rules missing fail_sql — add them to dq_rules.yaml manually: {todos}")
else:
    print("  All rules have fail_sql  ✓")

In [0]:
import pandas as pd
import yaml

FILE_PATH = "/Volumes/sales/mapping/mappings/master_mapping_bronze_silver_gold_pii.xlsx"
BASE      = "/Volumes/sales/mapping/mappings"

# ── Load existing SQL for complex tables (CTEs / window functions) ────────
# These cannot be auto-built from column-level mapping descriptions alone.
# Their SQL is preserved from the existing YAML and must be updated manually
# if the business logic changes.
try:
    with open(f"{BASE}/bronze_silver_transforms.yaml") as f:
        b2s_existing = yaml.safe_load(f)["tables"]
except FileNotFoundError:
    b2s_existing = {}

try:
    with open(f"{BASE}/silver_gold_transforms.yaml") as f:
        s2g_existing = yaml.safe_load(f)["tables"]
except FileNotFoundError:
    s2g_existing = {}

# Tables that require CTEs / window imputation / multi-step logic
B2S_COMPLEX = {
    "sales.silver.sellers",          # ROW_NUMBER dedup CTE
    "sales.silver.products",         # window avg imputation + derived volume/density
    "sales.silver.orders",           # derived date parts, flags, delay_days, etc.
    "sales.silver.geolocation",      # lat/lon dedup CTE
    "sales.silver.reviews_bad_rows", # quarantine filter
}

def col_expr(src_col, tgt_col, transform):
    """Return a SQL column expression from an Excel mapping row, or None for complex rows."""
    t = str(transform).strip() if pd.notna(transform) else ""
    s, tgt = str(src_col).strip(), str(tgt_col).strip()
    alias  = f" AS {tgt}" if s != tgt else ""

    SQL_PREFIXES = ("CAST(","LOWER(","UPPER(","TRIM(","COALESCE(",
                    "ROUND(","TO_DATE(","HOUR(","current_timestamp()")

    if not t or t in ("Pass-through", "Pass-through timestamp", "Pass through"):
        return f"{s}{alias}"
    if "SELECT DISTINCT" in t:
        return s  # DISTINCT applied at query level
    if any(t.startswith(p) for p in SQL_PREFIXES) or t == "current_timestamp()":
        return f"{t.split(';')[0].strip()}{alias}"
    if "CAST TO INT" in t.upper() and "null to 1" in t.lower():
        return f"COALESCE(CAST({s} AS INT), 1){alias}"
    if "CAST TO INT" in t.upper():
        return f"CAST({s} AS INT){alias}"
    if "CAST TO FLOAT" in t.upper():
        return f"CAST({s} AS FLOAT){alias}"
    if "DATE CONVERSION" in t.upper():
        return f"TO_DATE({s}){alias}"
    return None  # complex — skip (handled by COMPLEX override)

# ── Bronze → Silver ───────────────────────────────────────────────────────
b2s_df = pd.read_excel(FILE_PATH, sheet_name="Bronze_to_Silver", header=3).dropna(how="all")
b2s_tables, b2s_auto, b2s_carried = {}, 0, 0

for target, grp in b2s_df.groupby("Target Table"):
    target = str(target).strip()
    if target in B2S_COMPLEX:
        b2s_tables[target] = b2s_existing.get(target, f"-- TODO: add SQL for {target}")
        b2s_carried += 1
        continue

    src          = str(grp.iloc[0]["Source Table"]).strip()
    has_distinct = any("SELECT DISTINCT" in str(r["Transformation Logic"]) for _, r in grp.iterrows())
    exprs, wheres = [], []

    for _, row in grp.iterrows():
        expr = col_expr(row["Source Column"], row["Target Column"], row["Transformation Logic"])
        if expr:
            exprs.append(f"    {expr}")
        dq = str(row.get("DQ / Filter Rule", "")).strip()
        if dq.upper().startswith("WHERE ") and dq[6:].strip() not in wheres:
            wheres.append(dq[6:].strip())

    if not exprs:
        b2s_tables[target] = b2s_existing.get(target, f"-- TODO: add SQL for {target}")
        b2s_carried += 1
        continue

    distinct = "DISTINCT\n" if has_distinct else ""
    where    = f"\nWHERE {' AND '.join(wheres)}" if wheres else ""
    b2s_tables[target] = (f"CREATE OR REPLACE TABLE {target} AS\n"
                          f"SELECT {distinct}{',\n'.join(exprs)}\nFROM {src}{where}")
    b2s_auto += 1

with open(f"{BASE}/bronze_silver_transforms.yaml", "w") as f:
    yaml.dump({"tables": b2s_tables}, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"bronze_silver_transforms.yaml written  ✓  ({b2s_auto} auto-built | {b2s_carried} carried from existing)")

# ── Silver → Gold ───────────────────────────────────────────────────────
# Silver→Gold tables are full aggregation queries (CTEs, GROUP BY, JOINs).
# They are maintained in code below and written directly to the YAML.
# Update the SQL here if the Gold aggregation logic changes.
s2g_df     = pd.read_excel(FILE_PATH, sheet_name="Silver_to_Gold", header=3).dropna(how="all")
gold_tables = sorted(s2g_df["Target Gold Table"].dropna().unique())

# All gold tables use CTE/aggregation SQL — carry all from existing
s2g_tables  = {str(t).strip(): s2g_existing.get(str(t).strip(), f"-- TODO: add SQL for {t}")
               for t in gold_tables}

new_gold = [t for t in s2g_tables if "TODO" in s2g_tables[t]]

with open(f"{BASE}/silver_gold_transforms.yaml", "w") as f:
    yaml.dump({"tables": s2g_tables}, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f"silver_gold_transforms.yaml written  ✓  ({len(s2g_tables) - len(new_gold)} carried | {len(new_gold)} TODO)")
if new_gold:
    print(f"  ⚠  New gold tables with no SQL yet: {new_gold}")